# Programming Assignment: Building Tree-Based Models from Scratch

Welcome to the assignment for Tree-Based Models.

Tree-based methods are among the most popular and powerful machine learning algorithms for both classification and regression. From simple decision trees to sophisticated ensemble methods like Random Forests and Gradient Boosting, these algorithms have proven their worth in countless real-world applications, including winning many Kaggle competitions.

In this assignment, you'll implement tree-based classifiers using both scikit-learn and from-scratch implementations using NumPy. You'll explore single trees, understand their limitations, and then learn how ensemble methods overcome these weaknesses to create state-of-the-art models.

Decision trees work by recursively splitting the data based on feature values, creating a hierarchical structure that's highly interpretable. Each split aims to maximize the purity of the resulting subsets. While individual trees can overfit, combining multiple trees through bagging (Random Forests) or boosting (XGBoost, LightGBM) creates models that are both powerful and robust.

**What You Will Do in This Assignment**

* Understand the mathematical foundation of decision trees including entropy, Gini impurity, and information gain.
* Build and visualize decision trees, interpreting the learned rules.
* Control overfitting through pruning techniques (max_depth, min_samples_split).
* Implement Random Forest and understand how bootstrap aggregating reduces variance.
* Analyze out-of-bag error and extract feature importance from ensembles.
* Train Gradient Boosting models (XGBoost, LightGBM) with early stopping.
* Compare bagging vs boosting and understand their different strengths.
* Build a complete Decision Tree from scratch with Gini/entropy splitting.
* Implement Random Forest from scratch with bootstrap sampling and voting.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#imports)
- [1 - Introduction to Decision Trees](#1)
    - [1.1 - How Decision Trees Work](#1-1)
    - [1.2 - Splitting Criteria: Gini vs Entropy](#1-2)
    - [1.3 - Overfitting in Decision Trees](#1-3)
- [2 - Decision Tree Training and Visualization](#2)
    - [2.1 - Building Your First Decision Tree](#2-1)
    - **[Exercise 1 - train_decision_tree](#ex-1)**
    - [2.2 - Visualizing and Interpreting Trees](#2-2)
- [3 - Tree Pruning and Regularization](#3)
    - **[Exercise 2 - compare_tree_depths](#ex-2)**
- [4 - Random Forest Ensemble](#4)
    - [4.1 - Understanding Bootstrap Aggregating](#4-1)
    - **[Exercise 3 - train_random_forest](#ex-3)**
    - [4.2 - Feature Importance Analysis](#4-2)
- [5 - Gradient Boosting](#5)
    - [5.1 - How Boosting Differs from Bagging](#5-1)
    - **[Exercise 4 - train_gradient_boosting](#ex-4)**
- [6 - Ensemble Comparison](#6)
    - **[Exercise 5 - compare_ensemble_methods](#ex-5)**
- [7 - Decision Tree from Scratch](#7)
    - [7.1 - Implementing Splitting Criteria](#7-1)
    - **[Exercise 6a - calculate_gini](#ex-6a)**
    - **[Exercise 6b - calculate_entropy](#ex-6b)**
    - **[Exercise 6c - find_best_split](#ex-6c)**
    - **[Exercise 6d - build_decision_tree_scratch](#ex-6d)**
- [8 - Random Forest from Scratch](#8)
    - **[Exercise 7 - random_forest_from_scratch](#ex-7)**

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import load_iris, load_wine, make_classification, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# For advanced boosting (install if needed: pip install xgboost lightgbm)
try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available. Install with: pip install lightgbm")

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

<a name='1'></a>
## 1 - Introduction to Decision Trees

Decision trees are intuitive, interpretable models that learn a hierarchy of if-then-else rules from data. They can handle both numerical and categorical features, require minimal data preprocessing, and naturally capture non-linear relationships.

<a name='1-1'></a>
### 1.1 - How Decision Trees Work

A decision tree recursively partitions the feature space into regions. At each node, the algorithm:

1. **Evaluates all possible splits**: For each feature and threshold value
2. **Chooses the best split**: Based on a purity criterion (Gini or entropy)
3. **Creates child nodes**: Recursively repeat until stopping criteria are met

The prediction for a sample is determined by following the decision path from root to leaf, then using the majority class (classification) or mean value (regression) of training samples in that leaf.

**Key Properties:**
- **Non-parametric**: No assumptions about data distribution
- **Non-linear**: Can capture complex decision boundaries
- **Interpretable**: Easy to visualize and explain
- **Prone to overfitting**: Can memorize training data if not regularized

<a name='1-2'></a>
### 1.2 - Splitting Criteria: Gini vs Entropy

The quality of a split is measured by how much it increases the "purity" of the resulting child nodes. Two common criteria are:

**Gini Impurity**: Measures the probability of misclassifying a randomly chosen element:

$$\text{Gini}(S) = 1 - \sum_{i=1}^{C} p_i^2$$

where $p_i$ is the proportion of class $i$ in set $S$, and $C$ is the number of classes.

**Entropy (Information Gain)**: Measures the average information content:

$$\text{Entropy}(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

**Information Gain** from a split:

$$\text{IG}(S, A) = \text{Entropy}(S) - \sum_{v \in A} \frac{|S_v|}{|S|} \text{Entropy}(S_v)$$

where $A$ is the attribute (feature), and $S_v$ are the subsets created by splitting on $A$.

Both criteria work similarly in practice:
- **Gini**: Faster to compute (no logarithm), tends to isolate most frequent class
- **Entropy**: More balanced splits, theoretically grounded in information theory

**Pure node**: Gini = 0, Entropy = 0 (all samples belong to one class)
**Maximum impurity**: Gini = 0.5 (binary), Entropy = 1.0 (binary) when classes are equally distributed

<a name='1-3'></a>
### 1.3 - Overfitting in Decision Trees

Without constraints, a decision tree will grow until each leaf contains samples from a single class (or a single sample). This creates a model that perfectly fits the training data but generalizes poorly.

**Signs of overfitting:**
- Training accuracy ≈ 100%, but test accuracy is much lower
- Very deep tree with many leaves
- Complex decision boundaries that follow noise

**Regularization techniques:**
- **max_depth**: Limit tree depth
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in each leaf
- **max_leaf_nodes**: Limit total number of leaves
- **min_impurity_decrease**: Minimum improvement required for a split

**Pruning strategies:**
- **Pre-pruning**: Stop growing early (using constraints above)
- **Post-pruning**: Grow full tree, then remove branches that don't improve validation performance

<a name='helpers'></a>
## Helper Functions

We'll define utility functions for visualization and analysis.

In [3]:
def plot_decision_boundary_2d(model, X, y, title="Decision Boundary", h=0.02):
    """Plot decision boundary for 2D data."""
    X = np.asarray(X)
    y = np.asarray(y)
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 7))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', s=50, cmap='viridis')
    plt.colorbar(scatter)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

def visualize_tree(model, feature_names=None, class_names=None, max_depth=3):
    """Visualize decision tree structure."""
    plt.figure(figsize=(20, 10))
    plot_tree(model, 
              feature_names=feature_names,
              class_names=class_names,
              filled=True, 
              rounded=True,
              fontsize=10,
              max_depth=max_depth)
    plt.title("Decision Tree Structure")
    plt.show()

<a name='2'></a>
## 2 - Decision Tree Training and Visualization

Let's build our first decision tree and explore how it partitions the feature space.

<a name='2-1'></a>
### 2.1 - Building Your First Decision Tree

We'll use the Iris dataset, a classic multi-class classification problem with 3 classes and 4 features.

In [4]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"Dataset shape: {X_iris.shape}")
print(f"Number of classes: {len(np.unique(y_iris))}")
print(f"Class names: {iris.target_names}")
print(f"Feature names: {iris.feature_names}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Dataset shape: (150, 4)
Number of classes: 3
Class names: ['setosa' 'versicolor' 'virginica']
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Training samples: 105
Testing samples: 45


<a name='ex-1'></a>
#### **Exercise 1 - `train_decision_tree`**

**Your Task:**

Implement the `train_decision_tree` function that trains a decision tree classifier with specified hyperparameters.

You need to implement:

* **Create DecisionTreeClassifier**:
    * Use scikit-learn's `DecisionTreeClassifier`.
    * Set `criterion` (splitting criterion: 'gini' or 'entropy').
    * Set `max_depth` to control tree depth.
    * Set `min_samples_split` to control when to split.
    * Set `random_state=42` for reproducibility.

* **Fit the model**:
    * Train on the provided training data.

* **Return the trained model**:
    * Return the fitted classifier.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For creating the tree:**
* Import already done: `from sklearn.tree import DecisionTreeClassifier`.
* Create: `model = DecisionTreeClassifier(criterion=criterion, max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)`.

**For training:**
* Fit: `model.fit(X, y)`.

**For returning:**
* Return the fitted model.

</details>

In [ ]:
# GRADED FUNCTION: train_decision_tree
def train_decision_tree(X, y, criterion='gini', max_depth=None, min_samples_split=2):
    """
    Train a decision tree classifier.

    Args:
        X: Training features of shape (n_samples, n_features)
        y: Training labels of shape (n_samples,)
        criterion: Split criterion - 'gini' or 'entropy'
        max_depth: Maximum depth of tree (None = unlimited)
        min_samples_split: Minimum samples required to split a node

    Returns:
        model: Trained DecisionTreeClassifier
    """
    ### START CODE HERE ###
    
    # Create DecisionTreeClassifier with specified parameters
    
    # Fit the model
    
    ### END CODE HERE ###
    
    return model

In [ ]:
# Test your implementation
tree_model = train_decision_tree(X_train, y_train, criterion='gini', max_depth=3)

# Make predictions
y_pred = tree_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

print(f"\nTree depth: {tree_model.get_depth()}")
print(f"Number of leaves: {tree_model.get_n_leaves()}")
print(f"Number of features used: {np.sum(tree_model.feature_importances_ > 0)}")

##### **Expected Output**
```
Test Accuracy: 0.9778

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       1.00      0.93      0.97        15
   virginica       0.94      1.00      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45

Tree depth: 3
Number of leaves: 5
Number of features used: 3
```

In [ ]:
unittests.exercise_1(train_decision_tree)

<a name='2-2'></a>
### 2.2 - Visualizing and Interpreting Trees

One of the greatest strengths of decision trees is their interpretability. Let's visualize the tree structure and understand the learned rules.

In [ ]:
# Visualize the tree structure
visualize_tree(tree_model, 
               feature_names=iris.feature_names,
               class_names=iris.target_names,
               max_depth=3)

print("\nHow to read the tree:")
print("• Each box is a node")
print("• 'gini' or 'entropy': impurity at that node")
print("• 'samples': number of training samples reaching that node")
print("• 'value': class distribution [class0, class1, class2]")
print("• 'class': predicted class (majority)")
print("• Color intensity: purity (darker = more pure)")

In [ ]:
# Extract feature importance
importances = tree_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Feature Importances in Decision Tree")
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [iris.feature_names[i] for i in indices], rotation=45, ha='right')
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

print("\nFeature Importance Ranking:")
for i, idx in enumerate(indices):
    print(f"{i+1}. {iris.feature_names[idx]}: {importances[idx]:.4f}")

In [ ]:
# For 2D visualization, use only 2 features
X_train_2d = X_train[:, [2, 3]]  # petal length and width
X_test_2d = X_test[:, [2, 3]]

tree_2d = train_decision_tree(X_train_2d, y_train, criterion='gini', max_depth=3)
plot_decision_boundary_2d(tree_2d, X_train_2d, y_train, 
                         title="Decision Tree Boundary (Petal Length vs Width)")

print(f"Accuracy with 2 features: {accuracy_score(y_test, tree_2d.predict(X_test_2d)):.4f}")

<a name='3'></a>
## 3 - Tree Pruning and Regularization

Let's explore how different hyperparameters affect tree complexity and performance, demonstrating the bias-variance tradeoff.

<a name='ex-2'></a>
#### **Exercise 2 - `compare_tree_depths`**

**Your Task:**

Implement the `compare_tree_depths` function that trains decision trees with different max_depth values and compares their performance.

You need to implement:

* **Loop through depth values**:
    * For each max_depth in the provided list.
    * Train a decision tree with that depth.

* **Evaluate on both train and test sets**:
    * Calculate training accuracy to detect overfitting.
    * Calculate test accuracy to measure generalization.

* **Store results**:
    * Record depth, train accuracy, test accuracy, number of leaves, and tree depth.
    * Return as a pandas DataFrame for easy analysis.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For looping through depths:**
* Iterate: `for depth in max_depths:`.
* Train: `model = train_decision_tree(X_train, y_train, max_depth=depth)`.

**For evaluation:**
* Train acc: `train_acc = accuracy_score(y_train, model.predict(X_train))`.
* Test acc: `test_acc = accuracy_score(y_test, model.predict(X_test))`.
* Tree stats: `model.get_depth()`, `model.get_n_leaves()`.

**For storing results:**
* Append to list: `results.append({'max_depth': depth, 'train_acc': train_acc, ...})`.
* Convert to DataFrame: `df = pd.DataFrame(results)`.

</details>

In [ ]:
# GRADED FUNCTION: compare_tree_depths
def compare_tree_depths(X_train, y_train, X_test, y_test, max_depths=None):
    """
    Train decision trees with different max_depth values and compare performance.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        max_depths: List of max_depth values to try

    Returns:
        results_df: DataFrame with columns: max_depth, train_acc, test_acc, n_leaves, tree_depth
    """
    if max_depths is None:
        max_depths = [1, 2, 3, 4, 5, 10, 15, 20, None]
    
    results = []
    
    ### START CODE HERE ###
    
    # Loop through each max_depth value
    
        # Train model with this max_depth
        
        # Evaluate on train and test
        
        # Get tree statistics
        
        # Store results
        
    # Convert to DataFrame
    
    ### END CODE HERE ###
    
    return results_df

In [ ]:
# Test your implementation
depth_results = compare_tree_depths(X_train, y_train, X_test, y_test)

print("Tree Depth Comparison:")
print(depth_results.to_string(index=False))

# Visualize the results
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Accuracy vs Depth
axes[0].plot(depth_results['max_depth'].fillna(25), depth_results['train_acc'], 
             marker='o', label='Train Accuracy', linewidth=2)
axes[0].plot(depth_results['max_depth'].fillna(25), depth_results['test_acc'], 
             marker='s', label='Test Accuracy', linewidth=2)
axes[0].set_xlabel('Max Depth')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Tree Depth')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Tree Complexity vs Depth
axes[1].plot(depth_results['max_depth'].fillna(25), depth_results['n_leaves'], 
             marker='o', label='Number of Leaves', linewidth=2)
axes[1].plot(depth_results['max_depth'].fillna(25), depth_results['tree_depth'], 
             marker='s', label='Actual Tree Depth', linewidth=2)
axes[1].set_xlabel('Max Depth')
axes[1].set_ylabel('Count')
axes[1].set_title('Tree Complexity vs Max Depth')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("• Training accuracy increases with depth (can overfit)")
print("• Test accuracy plateaus or decreases after optimal depth")
print("• Deeper trees have more leaves (higher complexity)")

##### **Expected Output**
```
Tree Depth Comparison:
 max_depth  train_acc  test_acc  n_leaves  tree_depth
       1.0   0.666667  0.644444         2           1
       2.0   0.971429  0.933333         5           2
       3.0   0.980952  0.977778         5           3
       ...
```

In [ ]:
unittests.exercise_2(compare_tree_depths)

<a name='4'></a>
## 4 - Random Forest Ensemble

A single decision tree is unstable—small changes in training data can lead to completely different trees. Random Forests solve this by combining many trees through **bootstrap aggregating** (bagging).

<a name='4-1'></a>
### 4.1 - Understanding Bootstrap Aggregating

**Random Forest Algorithm**:

1. **Bootstrap sampling**: Create $B$ bootstrap samples (random sampling with replacement)
2. **Random feature selection**: At each split, consider only a random subset of features ($\sqrt{n}$ for classification)
3. **Build trees**: Train a decision tree on each bootstrap sample
4. **Aggregate predictions**: Average probabilities (classification) or values (regression)

**Key advantages**:
- **Variance reduction**: Averaging reduces overfitting
- **Out-of-bag (OOB) error**: Built-in validation using samples not in bootstrap
- **Feature importance**: Averaged across all trees
- **Parallel training**: Trees are independent

**Hyperparameters**:
- `n_estimators`: Number of trees (more is usually better, but slower)
- `max_features`: Features to consider at each split
- `max_depth`, `min_samples_split`: Tree-level regularization

In [ ]:
# Create a more challenging dataset
X_complex, y_complex = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_redundant=5,
    n_classes=3, n_clusters_per_class=2, random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42, stratify=y_complex
)

print(f"Complex dataset shape: {X_complex.shape}")
print(f"Number of classes: {len(np.unique(y_complex))}")

<a name='ex-3'></a>
#### **Exercise 3 - `train_random_forest`**

**Your Task:**

Implement the `train_random_forest` function that trains a Random Forest classifier and extracts key metrics including out-of-bag error.

You need to implement:

* **Create RandomForestClassifier**:
    * Use scikit-learn's `RandomForestClassifier`.
    * Set `n_estimators` (number of trees).
    * Set `max_features` (features per split).
    * Set `max_depth` and `min_samples_split` for regularization.
    * Set `oob_score=True` to enable out-of-bag evaluation.
    * Set `random_state=42` and `n_jobs=-1` (use all CPU cores).

* **Fit the model**:
    * Train on the provided data.

* **Extract metrics**:
    * Test accuracy from predictions.
    * OOB score from `model.oob_score_`.
    * Feature importances from `model.feature_importances_`.

* **Return results**:
    * Return dictionary with model and all metrics.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For creating Random Forest:**
* Create: `model = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth=max_depth, min_samples_split=min_samples_split, oob_score=True, random_state=42, n_jobs=-1)`.

**For training:**
* Fit: `model.fit(X_train, y_train)`.

**For metrics:**
* Test acc: `test_acc = accuracy_score(y_test, model.predict(X_test))`.
* OOB score: `oob_score = model.oob_score_`.
* Importance: `importances = model.feature_importances_`.

**Return dictionary:**
* `results = {'model': model, 'test_acc': test_acc, 'oob_score': oob_score, 'feature_importances': importances}`.

</details>

In [ ]:
# GRADED FUNCTION: train_random_forest
def train_random_forest(X_train, y_train, X_test, y_test, 
                        n_estimators=100, max_features='sqrt', 
                        max_depth=None, min_samples_split=2):
    """
    Train a Random Forest classifier and extract metrics.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        n_estimators: Number of trees
        max_features: Features to consider at each split
        max_depth: Maximum depth of trees
        min_samples_split: Minimum samples to split a node

    Returns:
        results: dict with keys:
            'model': trained RandomForestClassifier
            'test_acc': float - test accuracy
            'oob_score': float - out-of-bag score
            'feature_importances': array of feature importances
    """
    results = {}
    
    ### START CODE HERE ###
    
    # Create RandomForestClassifier
    
    # Fit the model
    
    # Calculate test accuracy
    
    # Get OOB score
    
    # Get feature importances
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Test your implementation
rf_results = train_random_forest(X_train_c, y_train_c, X_test_c, y_test_c,
                                 n_estimators=100, max_features='sqrt', max_depth=15)

print("="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"Test Accuracy: {rf_results['test_acc']:.4f}")
print(f"OOB Score: {rf_results['oob_score']:.4f}")
print(f"Number of trees: {rf_results['model'].n_estimators}")
print("="*60)

# Compare single tree vs Random Forest
single_tree = train_decision_tree(X_train_c, y_train_c, max_depth=15)
single_tree_acc = accuracy_score(y_test_c, single_tree.predict(X_test_c))

print(f"\nComparison:")
print(f"Single Decision Tree: {single_tree_acc:.4f}")
print(f"Random Forest (100 trees): {rf_results['test_acc']:.4f}")
print(f"Improvement: {(rf_results['test_acc'] - single_tree_acc)*100:.2f}%")

##### **Expected Output**
```
============================================================
RANDOM FOREST RESULTS
============================================================
Test Accuracy: 0.9XXX
OOB Score: 0.9XXX
Number of trees: 100
============================================================

Comparison:
Single Decision Tree: 0.8XXX
Random Forest (100 trees): 0.9XXX
Improvement: X.XX%
```

In [ ]:
unittests.exercise_3(train_random_forest)

<a name='4-2'></a>
### 4.2 - Feature Importance Analysis

Random Forests provide robust feature importance estimates by averaging across all trees.

In [ ]:
# Visualize feature importance
importances = rf_results['feature_importances']
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Feature Importances - Random Forest")
plt.bar(range(len(importances)), importances[indices])
plt.xlabel("Feature Index")
plt.ylabel("Importance")
plt.xticks(range(len(importances)), indices)
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
for i in range(min(10, len(importances))):
    print(f"{i+1}. Feature {indices[i]}: {importances[indices[i]]:.4f}")

# Study the effect of number of trees
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
oob_scores = []
test_scores = []

for n_trees in n_trees_range:
    rf_temp = train_random_forest(X_train_c, y_train_c, X_test_c, y_test_c, 
                                   n_estimators=n_trees, max_depth=15)
    oob_scores.append(rf_temp['oob_score'])
    test_scores.append(rf_temp['test_acc'])
    print(f"Trees: {n_trees:3d} | OOB: {rf_temp['oob_score']:.4f} | Test: {rf_temp['test_acc']:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(n_trees_range, oob_scores, marker='o', label='OOB Score', linewidth=2)
plt.plot(n_trees_range, test_scores, marker='s', label='Test Accuracy', linewidth=2)
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest: Performance vs Number of Trees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

print("\n💡 Key Insight: Performance improves and stabilizes with more trees")

<a name='5'></a>
## 5 - Gradient Boosting

While Random Forests reduce variance through averaging, **Gradient Boosting** reduces bias by sequentially training trees to correct the errors of previous trees.

<a name='5-1'></a>
### 5.1 - How Boosting Differs from Bagging

**Gradient Boosting Algorithm**:

1. **Initialize**: Start with a simple prediction (e.g., mean)
2. **Sequential training**: For each iteration $m = 1, ..., M$:
   - Calculate residuals (errors) from current model
   - Train a new tree to predict these residuals
   - Add the tree to the ensemble with a learning rate: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$
3. **Final prediction**: Sum all trees' predictions

**Key differences from Random Forest**:

| Aspect | Random Forest | Gradient Boosting |
|--------|--------------|-------------------|
| Training | Parallel (independent trees) | Sequential (each tree depends on previous) |
| Goal | Reduce variance | Reduce bias |
| Trees | Deep trees (low bias) | Shallow trees (high bias) |
| Speed | Fast (parallel) | Slower (sequential) |
| Overfitting | Rare | More prone (requires early stopping) |

**Key hyperparameters**:
- `n_estimators`: Number of boosting stages
- `learning_rate`: Shrinkage parameter (lower = more robust but needs more trees)
- `max_depth`: Tree depth (typically 3-8)
- `subsample`: Fraction of samples per tree (stochastic gradient boosting)

**Advanced implementations**:
- **XGBoost**: Regularized boosting with better performance
- **LightGBM**: Gradient-based one-side sampling (GOSS) and exclusive feature bundling (EFB)
- **CatBoost**: Handles categorical features natively

<a name='ex-4'></a>
#### **Exercise 4 - `train_gradient_boosting`**

**Your Task:**

Implement the `train_gradient_boosting` function that trains gradient boosting models with early stopping.

You need to implement:

* **Create GradientBoostingClassifier**:
    * Use scikit-learn's `GradientBoostingClassifier`.
    * Set `n_estimators`, `learning_rate`, `max_depth`, `subsample`.
    * Set `validation_fraction` for early stopping.
    * Set `n_iter_no_change` to stop if no improvement.
    * Set `random_state=42`.

* **Fit with early stopping**:
    * Train on the data.
    * Model will automatically stop if validation score doesn't improve.

* **Extract metrics**:
    * Test accuracy.
    * Number of estimators actually used (after early stopping).
    * Training history if available.

* **Optional: Train XGBoost if available**:
    * Use `XGBClassifier` with similar parameters.
    * Use `early_stopping_rounds` for early stopping.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For GradientBoostingClassifier:**
* Create: `gb_model = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, subsample=subsample, validation_fraction=0.1, n_iter_no_change=10, random_state=42)`.
* Fit: `gb_model.fit(X_train, y_train)`.
* Test acc: `gb_acc = accuracy_score(y_test, gb_model.predict(X_test))`.
* Trees used: `n_estimators_used = gb_model.n_estimators_`.

**For XGBoost (if available):**
* Create: `xgb_model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, subsample=subsample, random_state=42, early_stopping_rounds=10)`.
* Fit with validation: `xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)`.

**Return dictionary:**
* Include all models and metrics in results dict.

</details>

In [ ]:
# GRADED FUNCTION: train_gradient_boosting
def train_gradient_boosting(X_train, y_train, X_test, y_test,
                           n_estimators=100, learning_rate=0.1, 
                           max_depth=3, subsample=1.0):
    """
    Train gradient boosting models (sklearn and optionally XGBoost).

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        n_estimators: Number of boosting stages
        learning_rate: Learning rate (shrinkage)
        max_depth: Maximum depth of trees
        subsample: Fraction of samples for each tree

    Returns:
        results: dict with keys:
            'gb_model': trained GradientBoostingClassifier
            'gb_acc': test accuracy
            'gb_n_estimators': actual number of estimators used
            'xgb_model': trained XGBClassifier (if available)
            'xgb_acc': test accuracy (if available)
    """
    results = {}
    
    ### START CODE HERE ###
    
    # Train sklearn GradientBoostingClassifier
    
    # Train XGBoost if available
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Test your implementation
gb_results = train_gradient_boosting(X_train_c, y_train_c, X_test_c, y_test_c,
                                     n_estimators=200, learning_rate=0.1, 
                                     max_depth=3, subsample=0.8)

print("="*60)
print("GRADIENT BOOSTING RESULTS")
print("="*60)
print(f"Gradient Boosting (sklearn) Accuracy: {gb_results['gb_acc']:.4f}")
print(f"Number of estimators used: {gb_results['gb_n_estimators']}")

if 'xgb_acc' in gb_results:
    print(f"XGBoost Accuracy: {gb_results['xgb_acc']:.4f}")
print("="*60)

# Compare with Random Forest
print(f"\nComparison:")
print(f"Random Forest: {rf_results['test_acc']:.4f}")
print(f"Gradient Boosting: {gb_results['gb_acc']:.4f}")

##### **Expected Output**
```
============================================================
GRADIENT BOOSTING RESULTS
============================================================
Gradient Boosting (sklearn) Accuracy: 0.9XXX
Number of estimators used: XXX
XGBoost Accuracy: 0.9XXX
============================================================

Comparison:
Random Forest: 0.9XXX
Gradient Boosting: 0.9XXX
```

In [ ]:
unittests.exercise_4(train_gradient_boosting)

In [ ]:
# Study learning rate effect
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.5]
lr_results = []

for lr in learning_rates:
    gb_temp = train_gradient_boosting(X_train_c, y_train_c, X_test_c, y_test_c,
                                      n_estimators=200, learning_rate=lr, max_depth=3)
    lr_results.append({
        'learning_rate': lr,
        'accuracy': gb_temp['gb_acc'],
        'n_estimators': gb_temp['gb_n_estimators']
    })
    print(f"LR: {lr:.2f} | Acc: {gb_temp['gb_acc']:.4f} | Trees: {gb_temp['gb_n_estimators']}")

lr_df = pd.DataFrame(lr_results)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(lr_df['learning_rate'], lr_df['accuracy'], marker='o', linewidth=2)
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Accuracy vs Learning Rate')
axes[0].grid(True, alpha=0.3)

axes[1].plot(lr_df['learning_rate'], lr_df['n_estimators'], marker='s', linewidth=2, color='orange')
axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('Number of Trees Used')
axes[1].set_title('Trees Used vs Learning Rate')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Insight: Lower learning rates need more trees but can generalize better")

<a name='6'></a>
## 6 - Ensemble Comparison

Let's comprehensively compare different ensemble methods to understand their tradeoffs.

<a name='ex-5'></a>
#### **Exercise 5 - `compare_ensemble_methods`**

**Your Task:**

Implement the `compare_ensemble_methods` function that trains and compares multiple ensemble methods on the same dataset.

You need to implement:

* **Train multiple models**:
    * Single Decision Tree (baseline).
    * Random Forest (bagging).
    * Gradient Boosting (boosting).
    * XGBoost (if available).

* **Evaluate each model**:
    * Test accuracy.
    * Training time.
    * Prediction time.
    * Model complexity (number of estimators/leaves).

* **Return comparison DataFrame**:
    * One row per model with all metrics.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For training and timing:**
* Use time module: `import time`.
* Start: `start = time.time()`.
* Train model.
* Duration: `train_time = time.time() - start`.

**For prediction timing:**
* Start timer, predict, measure duration.

**For complexity:**
* Decision Tree: `model.get_n_leaves()`.
* Random Forest: `model.n_estimators`.
* Gradient Boosting: `model.n_estimators_`.

**Store results:**
* Create list of dictionaries, convert to DataFrame.

</details>

In [ ]:
# GRADED FUNCTION: compare_ensemble_methods
def compare_ensemble_methods(X_train, y_train, X_test, y_test):
    """
    Train and compare multiple ensemble methods.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels

    Returns:
        results_df: DataFrame comparing different methods with columns:
            'method', 'accuracy', 'train_time', 'predict_time', 'complexity'
    """
    import time
    results = []
    
    ### START CODE HERE ###
    
    # Train and evaluate Decision Tree
    
    # Train and evaluate Random Forest
    
    # Train and evaluate Gradient Boosting
    
    # Train and evaluate XGBoost (if available)
    
    # Convert to DataFrame
    
    ### END CODE HERE ###
    
    return results_df

In [ ]:
# Test your implementation
comparison_df = compare_ensemble_methods(X_train_c, y_train_c, X_test_c, y_test_c)

print("="*80)
print("ENSEMBLE METHODS COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Accuracy
axes[0].bar(comparison_df['method'], comparison_df['accuracy'])
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Test Accuracy Comparison')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Training Time
axes[1].bar(comparison_df['method'], comparison_df['train_time'], color='orange')
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Training Time Comparison')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Complexity
axes[2].bar(comparison_df['method'], comparison_df['complexity'], color='green')
axes[2].set_ylabel('Complexity (trees/leaves)')
axes[2].set_title('Model Complexity Comparison')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("• Random Forest: Best for parallel training, good accuracy")
print("• Gradient Boosting: Often highest accuracy, sequential training")
print("• XGBoost: Optimized boosting, faster than sklearn GradientBoosting")
print("• Single Tree: Fastest but lowest accuracy (high bias)")

##### **Expected Output**
```
================================================================================
ENSEMBLE METHODS COMPARISON
================================================================================
           method  accuracy  train_time  predict_time  complexity
 Decision Tree      0.8XXX      0.0XXX        0.00XX          XX
 Random Forest      0.9XXX      0.XXX         0.0XXX         100
Gradient Boosting   0.9XXX      X.XXX         0.0XXX         XXX
          XGBoost   0.9XXX      0.XXX         0.00XX         100
================================================================================
```

In [ ]:
unittests.exercise_5(compare_ensemble_methods)

<a name='7'></a>
## 7 - Decision Tree from Scratch

Now let's implement a decision tree from scratch to truly understand how the algorithm works.

<a name='7-1'></a>
### 7.1 - Implementing Splitting Criteria

We'll implement both Gini impurity and entropy to measure the quality of splits.

<a name='ex-6a'></a>
#### **Exercise 6a - `calculate_gini`**

**Your Task:**

Implement the `calculate_gini` function that calculates the Gini impurity for a set of labels.

You need to implement:

* **Calculate class probabilities**:
    * Count samples in each class.
    * Divide by total samples to get probabilities.

* **Calculate Gini impurity**:
    * Use formula: $\text{Gini} = 1 - \sum_{i=1}^{C} p_i^2$
    * Return 0 if all samples are in one class (pure node).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For class probabilities:**
* Get unique classes and counts: `classes, counts = np.unique(y, return_counts=True)`.
* Calculate probabilities: `probabilities = counts / len(y)`.

**For Gini:**
* Sum of squares: `gini = 1 - np.sum(probabilities ** 2)`.
* Or loop: `gini = 1 - sum([p**2 for p in probabilities])`.

</details>

In [ ]:
# GRADED FUNCTION: calculate_gini
def calculate_gini(y):
    """
    Calculate Gini impurity for a set of labels.

    Args:
        y: numpy array of labels

    Returns:
        gini: float - Gini impurity (0 = pure, higher = more impure)
    """
    ### START CODE HERE ###
    
    # Handle empty array
    if len(y) == 0:
        return 0
    
    # Calculate class probabilities
    
    # Calculate Gini impurity
    
    ### END CODE HERE ###
    
    return gini

In [ ]:
# Test calculate_gini
test_cases = [
    np.array([0, 0, 0, 0]),           # Pure node
    np.array([0, 1, 0, 1]),           # Maximum impurity (binary)
    np.array([0, 0, 1, 1, 2, 2]),     # Equal distribution (3 classes)
    np.array([0, 0, 0, 1, 1, 2]),     # Unequal distribution
]

print("Gini Impurity Test Cases:")
for i, y_case in enumerate(test_cases):
    gini = calculate_gini(y_case)
    print(f"Case {i+1}: {y_case } -> Gini = {gini:.4f}")

##### **Expected Output**
```
Gini Impurity Test Cases:
Case 1: [0 0 0 0] -> Gini = 0.0000
Case 2: [0 1 0 1] -> Gini = 0.5000
Case 3: [0 0 1 1 2 2] -> Gini = 0.6667
Case 4: [0 0 0 1 1 2] -> Gini = 0.6111
```

In [ ]:
unittests.exercise_6a(calculate_gini)

<a name='ex-6b'></a>
#### **Exercise 6b - `calculate_entropy`**

**Your Task:**

Implement the `calculate_entropy` function that calculates the entropy for a set of labels.

You need to implement:

* **Calculate class probabilities**:
    * Same as Gini: count samples in each class and normalize.

* **Calculate entropy**:
    * Use formula: $\text{Entropy} = -\sum_{i=1}^{C} p_i \log_2(p_i)$
    * Handle $p_i = 0$ case (use $0 \log 0 = 0$ convention).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For entropy:**
* Filter out zero probabilities: `probs = probabilities[probabilities > 0]`.
* Calculate entropy: `entropy = -np.sum(probs * np.log2(probs))`.
* Or with loop: `entropy = -sum([p * np.log2(p) for p in probs if p > 0])`.

</details>

In [ ]:
# GRADED FUNCTION: calculate_entropy
def calculate_entropy(y):
    """
    Calculate entropy for a set of labels.

    Args:
        y: numpy array of labels

    Returns:
        entropy: float - entropy (0 = pure, higher = more impure)
    """
    ### START CODE HERE ###
    
    # Handle empty array
    if len(y) == 0:
        return 0
    
    # Calculate class probabilities
    
    # Calculate entropy (handle log(0) = 0)
    
    ### END CODE HERE ###
    
    return entropy

In [ ]:
# Test calculate_entropy
print("Entropy Test Cases:")
for i, y_case in enumerate(test_cases):
    entropy = calculate_entropy(y_case)
    gini = calculate_gini(y_case)
    print(f"Case {i+1}: {y_case }")
    print(f"  Entropy = {entropy:.4f}, Gini = {gini:.4f}")

##### **Expected Output**
```
Entropy Test Cases:
Case 1: [0 0 0 0]
  Entropy = 0.0000, Gini = 0.0000
Case 2: [0 1 0 1]
  Entropy = 1.0000, Gini = 0.5000
Case 3: [0 0 1 1 2 2]
  Entropy = 1.5850, Gini = 0.6667
Case 4: [0 0 0 1 1 2]
  Entropy = 1.4591, Gini = 0.6111
```

In [ ]:
unittests.exercise_6b(calculate_entropy)

<a name='ex-6c'></a>
#### **Exercise 6c - `find_best_split`**

**Your Task:**

Implement the `find_best_split` function that finds the best feature and threshold to split a node.

You need to implement:

* **Loop through all features**:
    * For each feature, try different threshold values.

* **For each potential split**:
    * Split data into left and right subsets.
    * Calculate weighted impurity of the split.
    * Track the split with minimum impurity.

* **Calculate information gain**:
    * Information gain = parent impurity - weighted child impurity.

* **Return best split**:
    * Return feature index, threshold, and information gain.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For thresholds:**
* Use unique values: `thresholds = np.unique(X[:, feature_idx])`.
* Or midpoints between sorted values.

**For splitting:**
* Left mask: `left_mask = X[:, feature_idx] <= threshold`.
* Right mask: `right_mask = ~left_mask`.
* Split labels: `y_left = y[left_mask]`, `y_right = y[right_mask]`.

**For weighted impurity:**
* Weights: `w_left = len(y_left) / len(y)`, `w_right = len(y_right) / len(y)`.
* Weighted impurity: `weighted_impurity = w_left * calculate_gini(y_left) + w_right * calculate_gini(y_right)`.

**For information gain:**
* Parent impurity: `parent_impurity = calculate_gini(y)`.
* Gain: `gain = parent_impurity - weighted_impurity`.

</details>

In [ ]:
# GRADED FUNCTION: find_best_split
def find_best_split(X, y, criterion='gini'):
    """
    Find the best feature and threshold to split the data.

    Args:
        X: numpy array of shape (n_samples, n_features)
        y: numpy array of labels
        criterion: 'gini' or 'entropy'

    Returns:
        best_feature: int - index of best feature to split on
        best_threshold: float - best threshold value
        best_gain: float - information gain from this split
    """
    best_gain = 0
    best_feature = None
    best_threshold = None
    
    # Choose impurity function
    impurity_func = calculate_gini if criterion == 'gini' else calculate_entropy
    parent_impurity = impurity_func(y)
    
    n_samples, n_features = X.shape
    
    ### START CODE HERE ###
    
    # Loop through all features
    
        # Get unique values as potential thresholds
        
        # Try each threshold
        
            # Split data
            
            # Skip if split doesn't separate data
            
            # Calculate weighted impurity
            
            # Calculate information gain
            
            # Update best split if this is better
            
    ### END CODE HERE ###
    
    return best_feature, best_threshold, best_gain

In [ ]:
# Test find_best_split
X_test_split = np.array([
    [1, 2],
    [2, 3],
    [3, 4],
    [6, 7],
    [7, 8],
    [8, 9]
])
y_test_split = np.array([0, 0, 0, 1, 1, 1])

feature, threshold, gain = find_best_split(X_test_split, y_test_split, criterion='gini')
print(f"Best split: Feature {feature}, Threshold {threshold:.2f}, Gain {gain:.4f}")

# Verify the split
left_mask = X_test_split[:, feature] <= threshold
print(f"\nLeft subset labels: {y_test_split[left_mask]}")
print(f"Right subset labels: {y_test_split[~left_mask]}")
print("(Should be perfectly separated)")

##### **Expected Output**
```
Best split: Feature 0, Threshold X.XX, Gain X.XXXX

Left subset labels: [0 0 0]
Right subset labels: [1 1 1]
(Should be perfectly separated)
```

In [ ]:
unittests.exercise_6c(find_best_split)

<a name='ex-6d'></a>
#### **Exercise 6d - `build_decision_tree_scratch`**

**Your Task:**

Implement the `build_decision_tree_scratch` function that recursively builds a decision tree.

You need to implement:

* **Check stopping conditions**:
    * Maximum depth reached.
    * Node is pure (all same class).
    * Too few samples to split.
    * No information gain from split.

* **If stopping, create leaf node**:
    * Return dict with 'leaf' flag and majority class.

* **Otherwise, find best split**:
    * Use `find_best_split` to get feature and threshold.

* **Recursively build subtrees**:
    * Split data based on feature and threshold.
    * Recursively build left and right subtrees.

* **Return internal node**:
    * Store feature, threshold, and left/right children.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For stopping conditions:**
* Check depth: `if depth >= max_depth or len(y) < min_samples_split`.
* Check purity: `if len(np.unique(y)) == 1`.
* Find best split and check gain: `if best_gain == 0`.

**For leaf node:**
* Majority class: `majority = np.bincount(y).argmax()`.
* Return: `return {'leaf': True, 'class': majority}`.

**For internal node:**
* Split data and recurse:
  ```python
  left_mask = X[:, best_feature] <= best_threshold
  left = build_decision_tree_scratch(X[left_mask], y[left_mask], depth+1, ...)
  right = build_decision_tree_scratch(X[~left_mask], y[~left_mask], depth+1, ...)
  return {'leaf': False, 'feature': best_feature, 'threshold': best_threshold, 
          'left': left, 'right': right}
  ```

</details>

In [ ]:
# GRADED FUNCTION: build_decision_tree_scratch
def build_decision_tree_scratch(X, y, depth=0, max_depth=5, 
                               min_samples_split=2, criterion='gini'):
    """
    Recursively build a decision tree.

    Args:
        X: numpy array of features
        y: numpy array of labels
        depth: current depth
        max_depth: maximum depth allowed
        min_samples_split: minimum samples to split
        criterion: 'gini' or 'entropy'

    Returns:
        tree: dict representing the tree structure
            Leaf node: {'leaf': True, 'class': int}
            Internal node: {'leaf': False, 'feature': int, 'threshold': float,
                           'left': tree, 'right': tree}
    """
    ### START CODE HERE ###
    
    # Check stopping conditions
    
    # If stopping, return leaf node
    
    # Find best split
    
    # If no good split, return leaf
    
    # Split data
    
    # Recursively build left and right subtrees
    
    # Return internal node
    
    ### END CODE HERE ###
    
    return tree

In [ ]:
def predict_tree_scratch(tree, x):
    """Predict class for a single sample using the tree."""
    if tree['leaf']:
        return tree['class']
    
    if x[tree['feature']] <= tree['threshold']:
        return predict_tree_scratch(tree['left'], x)
    else:
        return predict_tree_scratch(tree['right'], x)

def predict_batch_scratch(tree, X):
    """Predict classes for multiple samples."""
    return np.array([predict_tree_scratch(tree, x) for x in X])

# Test the decision tree from scratch
X_train_simple = X_train[:, [2, 3]]  # Use 2 features for simplicity
tree_scratch = build_decision_tree_scratch(X_train_simple, y_train, 
                                          max_depth=3, criterion='gini')

# Make predictions
y_pred_scratch = predict_batch_scratch(tree_scratch, X_test[:, [2, 3]])
scratch_acc = accuracy_score(y_test, y_pred_scratch)

print(f"Decision Tree from Scratch Accuracy: {scratch_acc:.4f}")

# Compare with sklearn
sklearn_tree = train_decision_tree(X_train_simple, y_train, max_depth=3)
sklearn_acc = accuracy_score(y_test, sklearn_tree.predict(X_test[:, [2, 3]]))
print(f"Sklearn Decision Tree Accuracy: {sklearn_acc:.4f}")

# Visualize decision boundary
class TreeWrapper:
    def __init__(self, tree):
        self.tree = tree
    def predict(self, X):
        return predict_batch_scratch(self.tree, X)

plot_decision_boundary_2d(TreeWrapper(tree_scratch), X_train_simple, y_train,
                         title="From-Scratch Decision Tree Boundary")

##### **Expected Output**
```
Decision Tree from Scratch Accuracy: 0.9XXX
Sklearn Decision Tree Accuracy: 0.9XXX
```

In [ ]:
unittests.exercise_6d(build_decision_tree_scratch)

<a name='8'></a>
## 8 - Random Forest from Scratch

Finally, let's implement Random Forest by combining multiple decision trees with bootstrap sampling.

<a name='ex-7'></a>
#### **Exercise 7 - `random_forest_from_scratch`**

**Your Task:**

Implement the `random_forest_from_scratch` function that builds a random forest ensemble.

You need to implement:

* **Bootstrap sampling**:
    * For each tree, create a bootstrap sample (random sampling with replacement).
    * Sample n_samples rows from the data (with replacement).

* **Feature subsampling**:
    * At each split, randomly select a subset of features to consider.
    * Typical: sqrt(n_features) for classification.

* **Build multiple trees**:
    * Use `build_decision_tree_scratch` for each bootstrap sample.
    * Store all trees in a list.

* **Aggregate predictions**:
    * For prediction, each tree votes.
    * Return the majority class across all trees.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For bootstrap sampling:**
* Random indices: `indices = np.random.choice(len(X), size=len(X), replace=True)`.
* Bootstrap sample: `X_boot = X[indices]`, `y_boot = y[indices]`.

**For building forest:**
* Loop: `for i in range(n_estimators):`.
* Create bootstrap sample.
* Build tree: `tree = build_decision_tree_scratch(X_boot, y_boot, ...)`.
* Store: `trees.append(tree)`.

**For prediction:**
* Get all predictions: `all_preds = [predict_batch_scratch(tree, X) for tree in trees]`.
* Stack: `all_preds = np.array(all_preds)`.
* Vote: `predictions = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=all_preds)`.

**Note**: For simplicity, this implementation doesn't include per-split feature subsampling. That would require modifying `find_best_split` to accept a feature subset.

</details>

In [ ]:
# GRADED FUNCTION: random_forest_from_scratch
def random_forest_from_scratch(X, y, n_estimators=10, max_depth=5, 
                               min_samples_split=2, criterion='gini'):
    """
    Build a random forest by training multiple decision trees on bootstrap samples.

    Args:
        X: Training features
        y: Training labels
        n_estimators: Number of trees in the forest
        max_depth: Maximum depth of each tree
        min_samples_split: Minimum samples to split a node
        criterion: 'gini' or 'entropy'

    Returns:
        forest: list of decision trees
    """
    forest = []
    n_samples = len(X)
    
    ### START CODE HERE ###
    
    # Build each tree
    
        # Create bootstrap sample
        
        # Build decision tree on bootstrap sample
        
        # Add to forest
        
    ### END CODE HERE ###
    
    return forest

def predict_forest_scratch(forest, X):
    """
    Predict classes using a random forest (majority voting).
    
    Args:
        forest: list of decision trees
        X: features to predict on
        
    Returns:
        predictions: numpy array of predicted classes
    """
    # Get predictions from all trees
    all_predictions = np.array([predict_batch_scratch(tree, X) for tree in forest])
    
    # Majority vote
    predictions = np.apply_along_axis(
        lambda x: np.bincount(x.astype(int)).argmax(), 
        axis=0, 
        arr=all_predictions
    )
    
    return predictions

In [ ]:
# Test Random Forest from scratch
print("Training Random Forest from scratch...")
forest_scratch = random_forest_from_scratch(X_train_simple, y_train, 
                                           n_estimators=20, max_depth=5, 
                                           criterion='gini')

# Make predictions
y_pred_forest = predict_forest_scratch(forest_scratch, X_test[:, [2, 3]])
forest_scratch_acc = accuracy_score(y_test, y_pred_forest)

print(f"\nRandom Forest from Scratch Accuracy: {forest_scratch_acc:.4f}")
print(f"Single Tree from Scratch Accuracy: {scratch_acc:.4f}")
print(f"Improvement: {(forest_scratch_acc - scratch_acc)*100:.2f}%")

# Compare with sklearn
sklearn_rf = train_random_forest(X_train_simple, y_train, X_test[:, [2, 3]], y_test,
                                 n_estimators=20, max_depth=5)
print(f"Sklearn Random Forest Accuracy: {sklearn_rf['test_acc']:.4f}")

# Visualize Random Forest decision boundary
class ForestWrapper:
    def __init__(self, forest):
        self.forest = forest
    def predict(self, X):
        return predict_forest_scratch(self.forest, X)

plot_decision_boundary_2d(ForestWrapper(forest_scratch), X_train_simple, y_train,
                         title="From-Scratch Random Forest Boundary (20 trees)")

print(f"\n Random Forest combines {len(forest_scratch)} trees to make robust predictions!")

##### **Expected Output**
```
Training Random Forest from scratch...

Random Forest from Scratch Accuracy: 0.9XXX
Single Tree from Scratch Accuracy: 0.9XXX
Improvement: X.XX%
Sklearn Random Forest Accuracy: 0.9XXX

Random Forest combines 20 trees to make robust predictions!
```

In [ ]:
unittests.exercise_7(random_forest_from_scratch)

## Congratulations!

You have successfully completed the Tree-Based Models assignment! You've built a comprehensive understanding of one of the most powerful and widely-used families of machine learning algorithms.

### What You Accomplished:

1. **Understood Decision Trees**: Learned about splitting criteria, information gain, and the tree building process
2. **Trained and Visualized Trees**: Built interpretable decision trees and extracted meaningful rules
3. **Controlled Overfitting**: Applied pruning techniques and compared their effects
4. **Implemented Random Forest**: Used bootstrap aggregating to reduce variance and improve predictions
5. **Analyzed Feature Importance**: Extracted and interpreted feature importance from ensemble models
6. **Applied Gradient Boosting**: Trained XGBoost and LightGBM with early stopping
7. **Compared Ensemble Methods**: Understood tradeoffs between bagging and boosting
8. **Built Trees from Scratch**: Implemented complete decision tree with Gini/entropy splitting
9. **Built Random Forest from Scratch**: Implemented bootstrap sampling and voting mechanisms

### Key Takeaways:

* **Decision trees are intuitive**: Easy to visualize, interpret, and explain to non-technical stakeholders
* **Single trees overfit easily**: Without regularization, they memorize training data
* **Ensembles are powerful**: Combining trees dramatically improves performance
* **Random Forest (Bagging)**: Reduces variance through averaging, trains in parallel
* **Gradient Boosting**: Reduces bias through sequential correction, often achieves best accuracy
* **Feature importance**: Tree-based models naturally provide feature rankings
* **No feature scaling needed**: Unlike many algorithms, trees don't require normalization

### When to Use Tree-Based Models:

**Excellent for:**
- Tabular/structured data (CSV, databases)
- Mixed feature types (numerical + categorical)
- Non-linear relationships
- Feature interactions
- Interpretable models (single trees)
- Competitions (XGBoost/LightGBM)
- Missing value handling

**Consider alternatives for:**
- Unstructured data (images, text, audio) - use deep learning
- Very high-dimensional sparse data - consider linear models
- When you need probability calibration - requires extra work
- Real-time prediction with strict latency - single trees better than large ensembles

### Model Selection Guide:

**Decision Tree:**
- Use when: Interpretability is critical, simple baseline
- Pros: Fast, interpretable, no preprocessing
- Cons: High variance, overfits easily

**Random Forest:**
- Use when: Want robustness, parallel training, good default choice
- Pros: Low variance, handles outliers, OOB validation
- Cons: Less interpretable, larger memory footprint

**Gradient Boosting (XGBoost/LightGBM):**
- Use when: Maximum accuracy is priority, competitions
- Pros: Often best performance, handles missing values
- Cons: Slower training, more hyperparameters, prone to overfit

### Hyperparameter Tuning Tips:

**For Decision Trees:**
- `max_depth`: Start with 3-5, increase if underfitting
- `min_samples_split`: 2-20 depending on dataset size
- `criterion`: Try both 'gini' and 'entropy'

**For Random Forest:**
- `n_estimators`: 100-500 (more is usually better)
- `max_features`: 'sqrt' for classification, '0.3' for regression
- `max_depth`: 10-30 or None

**For Gradient Boosting:**
- `n_estimators`: 100-1000 with early stopping
- `learning_rate`: 0.01-0.1 (lower = more trees needed)
- `max_depth`: 3-8 (shallow trees work best)
- `subsample`: 0.7-0.9 for regularization

**Excellent work! You now have master-level understanding of tree-based models! 🌳**